## Data Processing

In this notebook, we process the data into a weekly format for weekly volatility estimations.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

data_dir = '../data/individual_ticker_data'
os.makedirs(data_dir, exist_ok=True)

In [2]:
tickers = ['SPY', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']

In [ ]:
# Part 1 start date (part 2 is 2016 so this works)
start_date = '2004-12-31' 
end_date = pd.to_datetime('today').strftime('%Y-%m-%d')

Loading data for tickers

In [ ]:
ticker_data = []
for ticker in tickers:
    df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False).reset_index()
    df = df.droplevel(level=1, axis=1)
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    df.rename(columns = {'date':'trade_date'}, inplace = True)
    df.insert(0, 'ticker', ticker)

    # Using arithmetic returns
    df['dly_ret'] = (df['close'] / df['close'].shift(1)) - 1
    df = df[1:].reset_index(drop = True)
    ticker_data.append(df)

### Wrangling Timeframes

In [ ]:
for df in ticker_data:
    # Assigning week number to each observation
    weekday = pd.to_datetime(df['trade_date']).dt.weekday
    week_num = []
    ix_week = 0
    week_num.append(ix_week)
    for ix in range(0, len(weekday) - 1):
        prev_day = weekday[ix]
        curr_day = weekday[ix + 1]
        if curr_day < prev_day:
            ix_week = ix_week + 1
        week_num.append(ix_week)
    np.array(week_num)
    df.insert(2, 'week_num', week_num)
    # Getting week start and end dates for each day
    df_start_end = \
    (
    df.groupby(['week_num'], as_index = False)[['trade_date']].agg([min, max])['trade_date']
    .rename(columns = {'min':'week_start', 'max':'week_end'})
    .reset_index()
    .rename(columns = {'index':'week_num'})
    )
    df = df.merge(df_start_end)

    # Rearranging
    cols = df.columns.tolist()
    cols = cols[:3] + cols[-2:] + cols[3:-2]
    df = df[cols]
    
    df.to_csv(os.path.join(data_dir, f'{df["ticker"][0]}.csv'), index=False)

C:\Users\rahul\AppData\Local\Temp\ipykernel_49012\2926314554.py:16: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  df.groupby(['week_num'], as_index = False)[['trade_date']].agg([min, max])['trade_date']
C:\Users\rahul\AppData\Local\Temp\ipykernel_49012\2926314554.py:16: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df.groupby(['week_num'], as_index = False)[['trade_date']].agg([min, max])['trade_date']
C:\Users\rahul\AppData\Local\Temp\ipykernel_49012\2926314554.py:16: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable 